In [ ]:
!pip install -q unsloth trl peft datasets

In [ ]:
import os
os.system("git lfs install")
os.system("git clone https://huggingface.co/Qwen/Qwen3-4B-Thinking-2507 /kaggle/working/model")

In [ ]:
from datasets import Dataset
import json
data = [json.loads(l) for l in open("/kaggle/input/bionic-data/grpo_train_ready.jsonl")]
dataset = Dataset.from_list(data)
print(f"Loaded {len(dataset)} examples")

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained("/kaggle/working/model", max_seq_length=1024, load_in_4bit=True, dtype=torch.bfloat16)
model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, target_modules=["q_proj","k_proj","v_proj","o_proj"], lora_dropout=0.05, bias="none", use_gradient_checkpointing="unsloth")

In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(output_dir="/kaggle/working/output", num_train_epochs=3, per_device_train_batch_size=4, gradient_accumulation_steps=4, learning_rate=1e-5, warmup_ratio=0.03, lr_scheduler_type="cosine", logging_steps=10, save_steps=50, bf16=True, beta=0.01, max_prompt_length=512, max_completion_length=512, report_to="none")
def format_reward(completions, **kwargs):
    return [1.0 if "<reasoning>" in c else 0.0 for c in completions]
def solution_reward(completions, **kwargs):
    return [1.0 if "<solution>" in c else 0.0 for c in completions]
trainer = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=[format_reward, solution_reward], args=training_args, train_dataset=dataset)
trainer.train()
model.save_pretrained("/kaggle/working/output")